# Causal Inference in Practice
## Week 8 — Instrumental Variables & Mendelian Randomization · Practice Notebook

> **Block III — Quasi-experimental designs**
>
> When the confounder can't be measured, find a valve that moves treatment and nothing else.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · The setup: an unmeasured confounder and a valve

Block II assumed every confounder was measured. This week we drop that: a confounder `U` drives both treatment `X` and outcome `Y`, and **we never get to see `U`**. Adjustment is therefore hopeless. Our rescue is an **instrument** `Z` — a variable that moves `X` and reaches `Y` *only* through `X`.

Throughout, the **true** causal effect of `X` on `Y` is a number we choose, so we can always check whether an estimate recovered it. We reuse the shared `RNG`.

In [ ]:
import statsmodels.api as sm

TRUE_BETA = 1.5          # the causal effect of X on Y (we get to know it)
n = 20_000

U = RNG.normal(size=n)                        # UNMEASURED confounder
Z = RNG.normal(size=n)                        # the instrument
X = 0.7*Z + 1.0*U + RNG.normal(size=n)        # first stage (Z->X) + U
Y = TRUE_BETA*X + 2.0*U + RNG.normal(size=n)  # U also hits Y; Z does NOT

print(f'true effect of X on Y = {TRUE_BETA}')
print('U is unmeasured, so we cannot adjust for it.')

Notice what is and isn't in the equations. `Z` enters `X` but **not** `Y` directly — that is the **exclusion restriction** baked into the simulation. `U` enters both `X` and `Y` — that is the confounding. Because `Z` does not depend on `U`, the instrument is **independent** of the outcome's other causes.

## 2 · OLS is biased; 2SLS recovers the truth

First, the naive regression of `Y` on `X`. Because `X` carries the hidden `U`, OLS is biased. Then **two-stage least squares**: regress `X` on `Z`, keep the fitted values `X̂` (the part of `X` the instrument explains, purged of `U`), and regress `Y` on `X̂`.

In [ ]:
# --- OLS: biased under confounding ---
ols_beta = sm.OLS(Y, sm.add_constant(X)).fit().params[1]

# --- 2SLS as two OLS stages ---
first  = sm.OLS(X, sm.add_constant(Z)).fit()   # stage 1: X on Z
Xhat   = first.fittedvalues                     # the Z-explained part of X
second = sm.OLS(Y, sm.add_constant(Xhat)).fit() # stage 2: Y on X-hat
iv_beta = second.params[1]

print(f'OLS  = {ols_beta:.3f}   (biased upward by U)')
print(f'2SLS = {iv_beta:.3f}   (truth = {TRUE_BETA})')
assert abs(iv_beta - TRUE_BETA) < 0.1, '2SLS should recover the truth'
assert ols_beta > TRUE_BETA + 0.2, 'OLS should be visibly biased here'

The first stage is what makes this work — and its strength is something we can **test**. The first-stage F statistic measures whether `Z` genuinely moves `X` (the **relevance** assumption).

In [ ]:
print(f'first-stage F = {first.fvalue:,.1f}')
assert first.fvalue > 100, 'this instrument is strong (good)'
print('F is huge -> a strong instrument. We trust the second stage.')

## 3 · The Wald estimator for a binary instrument

When the instrument is **binary**, 2SLS collapses to the **Wald estimator**: the instrument's effect on `Y` divided by its effect on `X`, i.e. the *reduced form over the first stage*. In means:

$$\hat\beta_{Wald}=\frac{\bar Y_{Z=1}-\bar Y_{Z=0}}{\bar X_{Z=1}-\bar X_{Z=0}}.$$

In [ ]:
TRUE2 = 2.0
Ub = RNG.normal(size=n)
Zb = RNG.binomial(1, 0.5, n)                   # binary instrument
Xb = 0.3 + 0.5*Zb + 0.8*Ub + RNG.normal(size=n)
Yb = TRUE2*Xb + 1.5*Ub + RNG.normal(size=n)

num = Yb[Zb==1].mean() - Yb[Zb==0].mean()      # reduced form (Z->Y)
den = Xb[Zb==1].mean() - Xb[Zb==0].mean()      # first stage  (Z->X)
wald = num / den

# 2SLS on the same data, for comparison
f2 = sm.OLS(Xb, sm.add_constant(Zb)).fit()
tsls = sm.OLS(Yb, sm.add_constant(f2.fittedvalues)).fit().params[1]

print(f'Wald  = {wald:.3f}')
print(f'2SLS  = {tsls:.3f}   (identical, as it should be)')
print(f'truth = {TRUE2}')
assert abs(wald - TRUE2) < 0.15
assert abs(wald - tsls) < 1e-6, 'Wald == 2SLS for a single binary Z'

### 🔧 Exercise 3.1 — write a reusable 2SLS function

Package the two-stage logic into a helper `two_stage(Z, X, Y)` that returns the IV slope. You'll reuse it for the rest of the notebook. Fill in the `# TODO`s — the skeleton already runs (the placeholders are `...`).

In [ ]:
def two_stage(Z, X, Y):
    """Return the 2SLS slope of Y on X, instrumented by Z."""
    first_stage = ...   # TODO: sm.OLS(X, sm.add_constant(Z)).fit()
    Xhat        = ...   # TODO: first_stage.fittedvalues
    second_stage = ...  # TODO: sm.OLS(Y, sm.add_constant(Xhat)).fit()
    return ...          # TODO: second_stage.params[1]

# Once filled in, this should print ~1.5 and pass the assert below.
# print(two_stage(Z, X, Y))

### ✅ Solution 3.1

In [ ]:
def two_stage(Z, X, Y):
    """Return the 2SLS slope of Y on X, instrumented by Z."""
    first_stage  = sm.OLS(X, sm.add_constant(Z)).fit()
    Xhat         = first_stage.fittedvalues
    second_stage = sm.OLS(Y, sm.add_constant(Xhat)).fit()
    return second_stage.params[1]

est = two_stage(Z, X, Y)
print(f'two_stage(Z, X, Y) = {est:.3f}   (truth {TRUE_BETA})')
assert abs(est - TRUE_BETA) < 0.1

def first_stage_F(Z, X):
    return sm.OLS(X, sm.add_constant(Z)).fit().fvalue

## 4 · Weak instruments: high variance, biased toward OLS

Relevance is not all-or-nothing. If `Z` barely moves `X`, the Wald ratio divides by a near-zero denominator and the estimate goes haywire. We compare a **strong** instrument (first-stage coefficient 0.7) with a **weak** one (0.03) by repeating each on many fresh samples and looking at the spread of the IV estimates.

In [ ]:
def make_iv_sample(m, z_coef):
    """One IV dataset with first-stage coefficient z_coef."""
    u = RNG.normal(size=m)
    z = RNG.normal(size=m)
    x = z_coef*z + 1.0*u + RNG.normal(size=m)
    y = TRUE_BETA*x + 2.0*u + RNG.normal(size=m)
    return z, x, y

strong, weak = [], []
for _ in range(300):
    strong.append(two_stage(*make_iv_sample(2000, 0.70)))
    weak.append(  two_stage(*make_iv_sample(2000, 0.03)))
strong, weak = np.array(strong), np.array(weak)

print(f'STRONG  mean={strong.mean():.2f}  std={strong.std():.2f}')
print(f'WEAK    mean={weak.mean():.2f}  std={weak.std():.2f}')
print(f'truth={TRUE_BETA}   OLS (biased) = {ols_beta:.2f}')
print(f'weak median = {np.median(weak):.2f}  '
      f'(pulled away from {TRUE_BETA} toward the OLS value {ols_beta:.2f})')

assert weak.std() > 3*strong.std(), 'weak IV must be far noisier'
assert abs(np.median(weak) - TRUE_BETA) > abs(strong.mean() - TRUE_BETA)
# the weak IV's center sits between the truth and the OLS bias
assert TRUE_BETA < np.median(weak) < ols_beta, 'weak IV is dragged toward OLS'

In [ ]:
# The diagnostic: a single weak sample's first-stage F is tiny.
zs, xs, ys = make_iv_sample(2000, 0.70)
zw, xw, yw = make_iv_sample(2000, 0.03)
print(f'strong first-stage F = {first_stage_F(zs, xs):8.1f}')
print(f'weak   first-stage F = {first_stage_F(zw, xw):8.1f}')
print('Rule of thumb: F < 10 -> weak instrument, do not trust the IV estimate.')

# A quick picture of the two sampling distributions.
fig, ax = plt.subplots()
ax.hist(np.clip(weak, -10, 12), bins=40, alpha=0.6, label='weak Z')
ax.hist(strong, bins=40, alpha=0.8, label='strong Z')
ax.axvline(TRUE_BETA, color='k', ls='--', label='truth = 1.5')
ax.set_title('IV estimates: strong vs weak instrument')
ax.set_xlabel('IV estimate'); ax.legend()
print('(figure created)')

The weak-instrument histogram is enormously wide and its center is dragged away from 1.5 toward the OLS bias. **This is why you report the first-stage F before interpreting anything.**

## 5 · LATE: an instrument identifies the complier effect

An instrument only moves *some* people. Split the population into **always-takers** (treated regardless), **never-takers** (untreated regardless), and **compliers** (treated iff encouraged). With monotonicity there are no **defiers**. We give compliers a *different* treatment effect from everyone else and show that IV recovers the **complier** effect — the **LATE** — not the population **ATE**.

In [ ]:
m = 60_000
Z = RNG.binomial(1, 0.5, m)                    # randomized encouragement
r = RNG.random(m)
# 50% compliers, 30% never-takers, 20% always-takers
kind = np.where(r < 0.50, 'complier',
        np.where(r < 0.80, 'never', 'always'))

# treatment status by type (note: compliers take treatment iff Z=1)
X = np.where(kind == 'complier', Z,
     np.where(kind == 'always', 1.0, 0.0))

# HETEROGENEOUS effects: compliers gain 3.0, others only 1.0
tau = np.where(kind == 'complier', 3.0, 1.0)
Y = RNG.normal(size=m) + tau*X                 # outcome = baseline + effect

ate  = tau.mean()                              # population ATE = E[tau]
late = (Y[Z==1].mean()-Y[Z==0].mean()) / (X[Z==1].mean()-X[Z==0].mean())

print(f'population ATE  = {ate:.3f}   (mix of 3.0 and 1.0)')
print(f'IV estimate    = {late:.3f}   <- targets the COMPLIER effect 3.0')
assert abs(late - 3.0) < 0.2, 'IV should recover the complier effect'
assert abs(late - ate) > 0.5, 'LATE is deliberately != ATE here'

In [ ]:
# The first-stage slope for a binary Z equals the SHARE of compliers.
complier_share = X[Z==1].mean() - X[Z==0].mean()
print(f'first-stage slope = {complier_share:.3f}  (true complier share 0.50)')
assert abs(complier_share - 0.50) < 0.03
print('A stronger first stage = more compliers = a more generalizable LATE.')

### 🔧 Exercise 5.1 — one-sided noncompliance

Now make it **one-sided**: there are no always-takers (you cannot get treatment unless encouraged) — only compliers and never-takers. Build 60% compliers / 40% never-takers, keep the complier effect at 3.0, and confirm IV still recovers 3.0. Fill in the `# TODO`s.

In [ ]:
Z2 = RNG.binomial(1, 0.5, m)
r2 = RNG.random(m)
kind2 = ...   # TODO: np.where(r2 < 0.60, 'complier', 'never')
X2 = ...      # TODO: compliers take treatment iff Z2==1, else 0.0
tau2 = ...    # TODO: 3.0 for compliers, 1.0 otherwise
# Y2 = RNG.normal(size=m) + tau2*X2
# late2 = (Y2[Z2==1].mean()-Y2[Z2==0].mean()) / \
#         (X2[Z2==1].mean()-X2[Z2==0].mean())
# print(late2)

### ✅ Solution 5.1

In [ ]:
Z2 = RNG.binomial(1, 0.5, m)
r2 = RNG.random(m)
kind2 = np.where(r2 < 0.60, 'complier', 'never')
X2 = np.where(kind2 == 'complier', Z2, 0.0)
tau2 = np.where(kind2 == 'complier', 3.0, 1.0)
Y2 = RNG.normal(size=m) + tau2*X2
late2 = (Y2[Z2==1].mean()-Y2[Z2==0].mean()) / \
        (X2[Z2==1].mean()-X2[Z2==0].mean())
print(f'one-sided LATE = {late2:.3f}   (complier effect 3.0)')
assert abs(late2 - 3.0) < 0.2

## 6 · Mendelian randomization: genotype as the instrument

Now the instrument is a **genetic variant** `G` — an allele count (0, 1, or 2). Because alleles are allocated at random at conception, `G` is plausibly independent of lifestyle confounders. We estimate the causal effect of an exposure (`LDL` cholesterol) on an outcome (`CHD`, heart disease). With a **valid** instrument the same 2SLS ratio recovers the truth.

In [ ]:
TRUE_MR = 0.8            # true causal effect of LDL on CHD
n = 40_000

C   = RNG.normal(size=n)                       # unmeasured confounder (lifestyle)
G   = RNG.binomial(2, 0.3, n).astype(float)    # genotype: 0/1/2 risk alleles
LDL = 1.0 + 0.5*G + 1.0*C + RNG.normal(size=n) # variant raises LDL
CHD = TRUE_MR*LDL + 1.5*C + RNG.normal(size=n) # G affects CHD ONLY via LDL

mr_valid = two_stage(G, LDL, CHD)
print(f'valid MR estimate = {mr_valid:.3f}   (truth {TRUE_MR})')
print(f'first-stage F     = {first_stage_F(G, LDL):,.0f}')
assert abs(mr_valid - TRUE_MR) < 0.1

## 7 · Pleiotropy breaks the exclusion restriction

**Pleiotropy** is the MR Achilles heel: the variant affects the outcome through a *second* pathway, not via the exposure. That is a direct arrow `G → CHD`, which violates **exclusion**. Watch the estimate break.

In [ ]:
# Same data, but now G ALSO hits CHD directly (pleiotropy):
CHD_pleio = TRUE_MR*LDL + 1.5*C + 0.9*G + RNG.normal(size=n)
#                                  ^^^^^ direct G->CHD path (exclusion violated)

mr_pleio = two_stage(G, LDL, CHD_pleio)
print(f'valid MR        = {mr_valid:.3f}   (truth {TRUE_MR})')
print(f'pleiotropic MR  = {mr_pleio:.3f}   (badly biased upward)')
assert mr_pleio > TRUE_MR + 0.3, 'pleiotropy should inflate the estimate'

With a **single** variant there is no way to tell a real effect from pleiotropy — the bias hides inside the ratio. The fix is to use **many** independent variants, which lets us *detect* pleiotropy.

## 8 · The robustness check: MR-Egger

With many variants, each contributes a (SNP→exposure, SNP→outcome) pair `(γⱼ, Γⱼ)`. Under valid instruments `Γⱼ ≈ β·γⱼ`, so a line through the origin has slope `β`. **MR-Egger** instead fits a line *with an intercept*: a non-zero intercept signals **directional pleiotropy**, and the slope is a pleiotropy-robust effect.

We simulate 30 variants, each with its own direct (pleiotropic) effect on the outcome, and compare the naive inverse-variance-weighted (IVW) slope with MR-Egger.

In [ ]:
TRUE_BETA_MR = 0.8
J = 30                                   # number of genetic variants
N = 60_000

gamma = RNG.uniform(0.10, 0.60, J)        # each variant's effect on exposure
alpha = RNG.uniform(0.05, 0.25, J)        # DIRECTIONAL pleiotropy (all positive)

Cc = RNG.normal(size=N)                                   # confounder
Gm = np.column_stack([RNG.binomial(2, 0.3, N).astype(float)
                      for _ in range(J)])
Exposure = 1.0 + Gm @ gamma + 1.0*Cc + RNG.normal(size=N)
Outcome  = TRUE_BETA_MR*Exposure + Gm @ alpha + 1.5*Cc + RNG.normal(size=N)

# per-variant SNP->exposure (gx) and SNP->outcome (gy) associations
gx = np.array([sm.OLS(Exposure, sm.add_constant(Gm[:, j])).fit().params[1]
               for j in range(J)])
gy = np.array([sm.OLS(Outcome,  sm.add_constant(Gm[:, j])).fit().params[1]
               for j in range(J)])
print(f'{J} variants simulated; pleiotropy is present (alpha > 0).')

In [ ]:
# IVW: regress gy on gx THROUGH THE ORIGIN (slope only).
ivw_slope = sm.OLS(gy, gx[:, None]).fit().params[0]

# MR-Egger: regress gy on gx WITH an intercept.
egger = sm.OLS(gy, sm.add_constant(gx)).fit()
egger_intercept, egger_slope = egger.params[0], egger.params[1]

print(f'truth            = {TRUE_BETA_MR}')
print(f'IVW slope        = {ivw_slope:.3f}   (biased up by pleiotropy)')
print(f'MR-Egger interc. = {egger_intercept:.3f}   (non-zero -> pleiotropy detected!)')
print(f'MR-Egger slope   = {egger_slope:.3f}   (closer to the truth)')

assert egger_intercept > 0.05, 'a non-zero intercept flags directional pleiotropy'
assert abs(egger_slope - TRUE_BETA_MR) < abs(ivw_slope - TRUE_BETA_MR)

In [ ]:
# The classic MR-Egger scatter: each point is a variant.
xx = np.linspace(0, gx.max()*1.05, 50)
fig, ax = plt.subplots()
ax.scatter(gx, gy, s=30, alpha=0.7, label='variants')
ax.plot(xx, ivw_slope*xx, label=f'IVW (slope {ivw_slope:.2f})')
ax.plot(xx, egger_intercept + egger_slope*xx,
        label=f'MR-Egger (int {egger_intercept:.2f})')
ax.axhline(0, color='grey', lw=0.8)
ax.set_xlabel('SNP -> exposure (gx)'); ax.set_ylabel('SNP -> outcome (gy)')
ax.set_title('MR-Egger: a non-zero intercept = directional pleiotropy')
ax.legend()
print('(figure created) The Egger line does NOT pass through the origin.')

### 🔧 Exercise 8.1 — does Egger's intercept vanish without pleiotropy?

Re-simulate the outcome with **no** pleiotropy (set every `alpha` to 0, i.e. drop the `Gm @ alpha` term) and recompute the MR-Egger intercept. It should now sit near zero, and IVW and Egger should agree. Fill in the `# TODO`s.

In [ ]:
alpha0 = np.zeros(J)                       # NO pleiotropy
Outcome0 = ...   # TODO: TRUE_BETA_MR*Exposure + Gm @ alpha0 + 1.5*Cc + RNG.normal(size=N)
# gy0 = np.array([sm.OLS(Outcome0, sm.add_constant(Gm[:, j])).fit().params[1]
#                 for j in range(J)])
# egger0 = sm.OLS(gy0, sm.add_constant(gx)).fit()
# print('intercept without pleiotropy:', egger0.params[0])

### ✅ Solution 8.1

In [ ]:
alpha0 = np.zeros(J)
Outcome0 = TRUE_BETA_MR*Exposure + Gm @ alpha0 + 1.5*Cc + RNG.normal(size=N)
gy0 = np.array([sm.OLS(Outcome0, sm.add_constant(Gm[:, j])).fit().params[1]
                for j in range(J)])
egger0 = sm.OLS(gy0, sm.add_constant(gx)).fit()
ivw0   = sm.OLS(gy0, gx[:, None]).fit().params[0]
print(f'intercept WITHOUT pleiotropy = {egger0.params[0]:.3f}  (~0)')
print(f'IVW slope = {ivw0:.3f}   Egger slope = {egger0.params[1]:.3f}'
      f'   (both ~{TRUE_BETA_MR})')
assert abs(egger0.params[0]) < 0.03, 'no pleiotropy -> intercept ~ 0'
assert abs(ivw0 - TRUE_BETA_MR) < 0.05

## Wrap-up & self-check

- **OLS is biased** under unmeasured confounding; **2SLS** (two OLS stages) and the **Wald** ratio recover the truth when the instrument is valid.
- The four assumptions: **relevance** (testable — first-stage F), **exclusion** and **independence** (untestable — argued), and **monotonicity** (names the estimand).
- IV identifies the **LATE** — the **complier** effect — which need not equal the ATE.
- A **weak instrument** (small first-stage F) gives a high-variance estimate biased back toward OLS. Always report the F.
- In **Mendelian randomization**, genotype is the instrument; **pleiotropy** breaks exclusion, and **MR-Egger**'s intercept detects directional pleiotropy.

**You're ready for Week 9** (regression discontinuity) if you can implement 2SLS from memory, read a first-stage F, and explain why pleiotropy biases an MR estimate. And good luck on the **midterm (Weeks 1–7)**.